# 01. Quantum States, Wavefunctions & Spatial Discretization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/computational-quantum-optics/blob/main/module_0_qm_fundamentals/01_quantum_states_and_wavefunctions.ipynb)

* **Module:** 0 — Quantum Mechanics Fundamentals
* **Difficulty:** Beginner
* **Estimated Time:** 35 minutes
* **Prerequisites:** Introductory Python, basic calculus (integrals), and elementary linear algebra.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Discretize continuous 1D spatial domains** and represent quantum wavefunctions $\psi(x)$ on a computational grid.
2. **Enforce the Born normalization condition** using numerical quadrature ($\int |\psi(x)|^2 dx = 1$).
3. **Simulate quantum superpositions** and isolate the physical quantum interference term from classical probability sums.
4. **Calculate quantum observables** (expectation values $\langle x angle, \langle x^2 angle$ and position uncertainty $\Delta x$) via numerical integration.
5. **Bridge continuous wave mechanics to discrete state vectors** through the concept of the two-level quantum bit (Qubit).


---
## 0. Setup & Dependencies

In Google Colab, standard packages are pre-installed. We set up publication-ready styling for our visualizations.


In [ ]:
# Google Colab installation (uncomment if running on a fresh cloud instance)
# !pip install -q numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

# Configure clean, accessible plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

print("Environment initialized successfully!")


---
## 1. The Continuous Wavefunction & Born's Rule

In quantum mechanics, the complete state of a particle in one spatial dimension is encoded in a complex-valued scalar field called the **wavefunction**, denoted by $\psi(x)$. 

Unlike a classical trajectory $x(t)$ that assigns a definite position to a particle at every instant, $\psi(x)$ is a **probability amplitude**. Max Born (1926) established that the modulus squared of the amplitude gives the physical **probability density** $P(x)$ of discovering the particle in an infinitesimal interval $[x, x + dx]$:

$$P(x) = |\psi(x)|^2 = \psi^*(x)\,\psi(x)$$

Since the total probability of finding the particle anywhere along the 1D real line must equal unity, any physically admissible state must satisfy the **normalization condition**:

$$\int_{-\infty}^{\infty} |\psi(x)|^2\,dx = 1$$

Mathematically, this restricts physical wavefunctions to the Hilbert space of square-integrable functions, denoted as $\mathcal{H} = L^2(\mathbb{R})$.

### The 1D Gaussian Wave Packet
The most ubiquitous state in quantum optics and wave mechanics is the **Gaussian wave packet**. It represents a localized particle centered at mean position $x_0$ with spatial standard deviation $\sigma$ and central wavevector $k_0$ (corresponding to average momentum $p_0 = \hbar k_0$):

$$\psi(x) = \left(\frac{1}{2\pi \sigma^2}\right)^{1/4} \exp\left( -\frac{(x - x_0)^2}{4\sigma^2} + i k_0 x \right)$$

Notice that $\psi(x)$ is inherently complex:
* The **envelope** $\exp\left(-\frac{(x - x_0)^2}{4\sigma^2}\right)$ governs where the particle can be found.
* The **phase factor** $e^{i k_0 x} = \cos(k_0 x) + i \sin(k_0 x)$ carries spatial oscillations reflecting momentum.


In [ ]:
def gaussian_wavepacket(x: np.ndarray, x0: float = 0.0, sigma: float = 1.0, k0: float = 0.0) -> np.ndarray:
    """
    Computes a normalized 1D Gaussian wavepacket on a spatial coordinate grid.
    
    Parameters:
        x (np.ndarray): 1D array of spatial grid coordinates.
        x0 (float): Mean position (center of envelope).
        sigma (float): Spatial width / standard deviation of the envelope.
        k0 (float): Central wavenumber (momentum p = hbar * k0).
        
    Returns:
        np.ndarray: Complex array representing the normalized wavefunction psi(x).
    """
    # Normalization prefactor: (1 / (2 * pi * sigma^2))^(1/4)
    prefactor = (1.0 / (2.0 * np.pi * sigma**2))**0.25
    
    # Gaussian envelope
    envelope = np.exp(- (x - x0)**2 / (4.0 * sigma**2))
    
    # Plane-wave phase oscillation
    phase = np.exp(1j * k0 * x)
    
    return prefactor * envelope * phase

def check_normalization(x: np.ndarray, psi: np.ndarray) -> float:
    """
    Computes the total probability integral: \\int |psi(x)|^2 dx using Simpson's composite rule.
    """
    prob_density = np.abs(psi)**2
    total_prob = integrate.simpson(y=prob_density, x=x)
    return total_prob

# Define a discrete spatial grid x in [-10, 10] with N = 1000 points
L = 10.0
N = 1000
x = np.linspace(-L, L, N)
dx = x[1] - x[0]

# Generate a Gaussian wavepacket centered at x0 = -1.0 with k0 = 4.0
psi_demo = gaussian_wavepacket(x, x0=-1.0, sigma=1.2, k0=4.0)
norm_val = check_normalization(x, psi_demo)

print(f"Spatial grid step dx = {dx:.4f}")
print(f"Calculated norm integral ∫|ψ(x)|² dx = {norm_val:.8f}")


In [ ]:
# Visualization of the Wavefunction and its Probability Density
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot Real and Imaginary parts alongside the envelope
ax1.plot(x, np.real(psi_demo), label=r"$\mathrm{Re}[\psi(x)]$", color="#1f77b4", lw=1.8)
ax1.plot(x, np.imag(psi_demo), label=r"$\mathrm{Im}[\psi(x)]$", color="#ff7f0e", lw=1.5, ls="--")
ax1.plot(x, np.abs(psi_demo), label=r"$|\psi(x)|$ (Envelope)", color="#2ca02c", lw=2.0)
ax1.set_title("Wavefunction Components & Phase")
ax1.set_xlabel("Position $x$")
ax1.set_ylabel("Amplitude")
ax1.set_xlim(-6, 6)
ax1.legend(loc="upper right")

# Plot the Physical Probability Density
prob_density = np.abs(psi_demo)**2
ax2.plot(x, prob_density, label=r"$P(x) = |\psi(x)|^2$", color="#d62728", lw=2.0)
ax2.fill_between(x, prob_density, color="#d62728", alpha=0.25, label="Total Probability Area = 1.0")
ax2.set_title("Born Probability Density")
ax2.set_xlabel("Position $x$")
ax2.set_ylabel("Probability Density $P(x)$")
ax2.set_xlim(-6, 6)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()


> **Physical Insight:** Notice in the left panel that while $\text{Re}[\psi(x)]$ and $\text{Im}[\psi(x)]$ oscillate rapidly back and forth due to the momentum factor $e^{i k_0 x}$, the physical probability density $P(x)$ on the right is a smooth, stationary Gaussian curve. The phase oscillations represent internal quantum momentum, yet the spatial discovery probability remains entirely insensitive to global phase rotations $\psi(x) \to e^{i\theta}\psi(x)$.


---
## 2. The Superposition Principle & Quantum Interference

The foundational axiom of quantum mechanics is the **superposition principle**: if $\psi_1(x)$ and $\psi_2(x)$ are two valid quantum states, any linear combination 

$$\psi_{\text{total}}(x) = c_1 \psi_1(x) + c_2 \psi_2(x) \quad (c_1, c_2 \in \mathbb{C})$$

is also an admissible quantum state.

When we observe probabilities, a purely quantum phenomenon emerges. In classical probability theory, if an event can occur via two mutually exclusive channels, the joint probability is simply the direct sum of the individual probabilities: $P_{\text{classical}} = P_1 + P_2$. 

In quantum physics, **amplitudes add before squaring**:

$$P_{\text{quantum}}(x) = |\psi_{\text{total}}(x)|^2 = |c_1 \psi_1(x) + c_2 \psi_2(x)|^2$$

Expanding this product reveals three terms:

$$P_{\text{quantum}}(x) = |c_1|^2 |\psi_1(x)|^2 + |c_2|^2 |\psi_2(x)|^2 + \underbrace{2\,\text{Re}\left[c_1^* c_2\,\psi_1^*(x)\psi_2(x)\right]}_{\text{Quantum Interference Term } \mathcal{I}(x)}$$

* When $\mathcal{I}(x) > 0$, we observe **constructive interference** (probability exceeds the classical sum).
* When $\mathcal{I}(x) < 0$, we observe **destructive interference** (probability is lower than the classical sum, potentially dropping to zero).


In [ ]:
# Define two separated wavepackets (mimicking a double-slit or beam splitter output)
d = 1.8         # Separation half-distance
sigma_sub = 0.9 # Width of each wavepacket

psi_1 = gaussian_wavepacket(x, x0=-d, sigma=sigma_sub, k0=1.0)
psi_2 = gaussian_wavepacket(x, x0=+d, sigma=sigma_sub, k0=-1.0)

# Create an equal superposition with adjustable relative phase phi
phi = np.pi  # Relative phase (try 0 vs pi)
c1 = 1.0 / np.sqrt(2.0)
c2 = np.exp(1j * phi) / np.sqrt(2.0)

psi_super = c1 * psi_1 + c2 * psi_2

# Compute the classical sum vs quantum probability
P_classical = np.abs(c1)**2 * np.abs(psi_1)**2 + np.abs(c2)**2 * np.abs(psi_2)**2
P_quantum = np.abs(psi_super)**2
Interference_term = 2.0 * np.real(np.conj(c1 * psi_1) * (c2 * psi_2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, P_classical, label="Classical Sum: $|c_1|^2 P_1 + |c_2|^2 P_2$", color="black", ls="--", lw=1.8)
ax.plot(x, P_quantum, label=f"Quantum Superposition $P_{{quantum}}$ (Phase $\\phi = {phi/np.pi:.1f}\pi$)", color="#d62728", lw=2.2)
ax.fill_between(x, Interference_term, 0, where=(Interference_term > 0), color="blue", alpha=0.2, label="Constructive Interference (+)")
ax.fill_between(x, Interference_term, 0, where=(Interference_term < 0), color="red", alpha=0.2, label="Destructive Interference (-)")

ax.set_title("Quantum Interference vs. Classical Probability Addition")
ax.set_xlabel("Position $x$")
ax.set_ylabel("Probability Density")
ax.set_xlim(-6, 6)
ax.legend(loc="upper right")
plt.show()


> **Physical Insight:** When the relative phase is $\phi = \pi$, destructive interference forces the probability density strictly to zero at the origin $x = 0$, creating a **nodal point** even though both independent wavepackets have non-zero probability amplitude there! This phenomenon is the spatial foundation of the **Hong-Ou-Mandel effect** and **Schrödinger Cat states** that we will explore in Modules 1, 2, and 5.


---
## 3. Quantum Observables, Expectation Values & Uncertainty

Physical quantities (position, momentum, energy) are represented in quantum mechanics by linear, **Hermitian operators**. Because measurement outcomes are inherently statistical, the **expectation value** $\langle \hat{A} \rangle$ corresponds to the arithmetic mean of an infinite ensemble of identical measurements performed on identically prepared states $\psi(x)$.

For the continuous position observable $\hat{x}$:

$$\langle \hat{x} \rangle = \int_{-\infty}^{\infty} \psi^*(x)\,\hat{x}\,\psi(x)\,dx = \int_{-\infty}^{\infty} x\,|\psi(x)|^2\,dx$$

$$\langle \hat{x}^2 \rangle = \int_{-\infty}^{\infty} x^2\,|\psi(x)|^2\,dx$$

The **quantum uncertainty** (or standard deviation) $\Delta x$ quantifies the statistical spread around the mean value:

$$\Delta x = \sqrt{\langle \hat{x}^2 \rangle - \langle \hat{x} \rangle^2}$$

Let us numerically implement these integrals and verify them against known analytical formulas for Gaussian wavepackets.


In [ ]:
def compute_position_statistics(x: np.ndarray, psi: np.ndarray) -> dict:
    """
    Calculates the first and second moments of position, along with the spatial uncertainty Delta x.
    """
    prob = np.abs(psi)**2
    
    # Normalization check
    norm = integrate.simpson(y=prob, x=x)
    prob_normalized = prob / norm
    
    # Expectation value <x> = \int x |psi(x)|^2 dx
    mean_x = integrate.simpson(y=x * prob_normalized, x=x)
    
    # Expectation value <x^2> = \int x^2 |psi(x)|^2 dx
    mean_x2 = integrate.simpson(y=(x**2) * prob_normalized, x=x)
    
    # Variance and Standard Deviation
    var_x = mean_x2 - mean_x**2
    delta_x = np.sqrt(max(0.0, var_x))
    
    return {
        "mean_x": mean_x,
        "mean_x2": mean_x2,
        "delta_x": delta_x
    }

# Test on a Gaussian wavepacket with target x0 = 2.5 and target sigma = 1.35
target_x0 = 2.5
target_sigma = 1.35
test_psi = gaussian_wavepacket(x, x0=target_x0, sigma=target_sigma, k0=2.0)

stats = compute_position_statistics(x, test_psi)

print("--- Numerical vs Analytical Verification ---")
print(f"Computed <x>       = {stats['mean_x']:.6f}   | Theoretical x0    = {target_x0:.6f}")
print(f"Computed Δx        = {stats['delta_x']:.6f}   | Theoretical σ     = {target_sigma:.6f}")
print(f"Absolute error Δx = {abs(stats['delta_x'] - target_sigma):.2e}")


---
## 4. From Continuous Fields to Discrete Spaces: The Qubit Concept

While spatial wavefunctions live in an infinite-dimensional continuous function space $L^2(\mathbb{R})$, many critical quantum systems inhabit finite-dimensional discrete vector spaces $\mathbb{C}^d$. 

The fundamental unit of quantum information is the **two-level quantum bit (Qubit)** living in $\mathbb{C}^2$. The orthonormal basis states are represented in Dirac notation as:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Any arbitrary normalized pure qubit state $|\psi\rangle$ can be expressed as a linear superposition:

$$|\psi\rangle = \alpha |0\rangle + \beta |1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \quad |\alpha|^2 + |\beta|^2 = 1$$

Because the global phase $e^{i\gamma}$ is physically unobservable, we can parameterize any pure qubit using two real angles $\theta \in [0, \pi]$ and $\phi \in [0, 2\pi)$ on the surface of a unit sphere called the **Bloch Sphere**:

$$|\psi(\theta, \phi)\rangle = \cos\left(\frac{\theta}{2}\right)|0\rangle + e^{i\phi}\sin\left(\frac{\theta}{2}\right)|1\rangle$$

### Comparison: Continuous Wavefunction vs. Discrete Qubit

| Property | Continuous Wavefunction $\psi(x)$ | Discrete Qubit $|\psi\rangle$ |
| :--- | :--- | :--- |
| **Hilbert Space** | Infinite-dimensional $L^2(\mathbb{R})$ | Two-dimensional $\mathbb{C}^2$ |
| **Inner Product** | $\langle \phi | \psi \rangle = \int_{-\infty}^{\infty} \phi^*(x)\psi(x)\,dx$ | $\langle \phi | \psi \rangle = \phi_0^* \psi_0 + \phi_1^* \psi_1$ |
| **Completeness** | $\int_{-\infty}^{\infty} |x\rangle\langle x|\,dx = \mathbb{I}$ | $|0\rangle\langle 0| + |1\rangle\langle 1| = \mathbb{I}$ |
| **Probability Density** | $P(x) = |\psi(x)|^2$ | $P(0) = |\alpha|^2, \quad P(1) = |\beta|^2$ |


In [ ]:
# Visualization of Qubit State Amplitudes and Vector Representation
def plot_qubit_amplitudes(theta: float, phi: float):
    """
    Computes alpha and beta for a qubit state and displays their probabilities and relative phase.
    """
    alpha = np.cos(theta / 2.0)
    beta = np.exp(1j * phi) * np.sin(theta / 2.0)
    
    p0 = np.abs(alpha)**2
    p1 = np.abs(beta)**2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # 1. Measurement probabilities
    states = [r"$|0\rangle$", r"$|1\rangle$"]
    probs = [p0, p1]
    ax1.bar(states, probs, color=["#1f77b4", "#ff7f0e"], width=0.45, alpha=0.85)
    ax1.set_ylim(0, 1.1)
    ax1.set_ylabel("Measurement Probability")
    ax1.set_title(rf"Born Probabilities ($\theta={theta/np.pi:.2f}\pi, \phi={phi/np.pi:.2f}\pi$)")
    for i, p in enumerate(probs):
        ax1.text(i, p + 0.03, f"{p:.3f}", ha="center", fontweight="bold")
        
    # 2. Complex amplitude Argand plane for beta (with alpha real)
    ax2.axhline(0, color="gray", lw=1)
    ax2.axvline(0, color="gray", lw=1)
    ax2.quiver(0, 0, np.real(beta), np.imag(beta), angles="xy", scale_units="xy", scale=1, 
               color="#ff7f0e", width=0.02, label=r"$\beta = e^{i\phi}\sin(\theta/2)$")
    ax2.set_xlim(-1.1, 1.1)
    ax2.set_ylim(-1.1, 1.1)
    ax2.set_aspect("equal")
    ax2.set_xlabel(r"$\mathrm{Re}[\beta]$")
    ax2.set_ylabel(r"$\mathrm{Im}[\beta]$")
    ax2.set_title(r"Complex Amplitude $\beta$ Phase in $\mathbb{C}$")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# Example: An equal superposition state |+> = (|0> + |1>) / sqrt(2) -> theta = pi/2, phi = pi/4
plot_qubit_amplitudes(theta=np.pi/2, phi=np.pi/4)


---
## 5. Interactive Playground: Quantum Interference Explorer

Use the interactive sliders below to explore how the **separation distance $d$**, **relative phase $\phi$**, and **packet width $\sigma$** shape quantum interference. 

Notice how shifting the phase continuously relocates the fringes between constructive and destructive interference.


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def interactive_interference(d=1.8, phi_factor=1.0, sigma=1.0, k0=1.0):
    phi = phi_factor * np.pi
    
    psi_A = gaussian_wavepacket(x, x0=-d, sigma=sigma, k0=k0)
    psi_B = gaussian_wavepacket(x, x0=+d, sigma=sigma, k0=-k0)
    
    # Superposition: psi = 1/sqrt(2) * (psi_A + e^(i*phi) * psi_B)
    psi_tot = (psi_A + np.exp(1j * phi) * psi_B) / np.sqrt(2.0)
    
    # Normalize numerically for varying overlaps
    norm = np.sqrt(integrate.simpson(y=np.abs(psi_tot)**2, x=x))
    psi_tot /= norm
    
    P_tot = np.abs(psi_tot)**2
    P_indiv = 0.5 * (np.abs(psi_A)**2 + np.abs(psi_B)**2)
    
    plt.figure(figsize=(10, 4.5))
    plt.plot(x, P_indiv, 'k--', label="Classical Sum (No Interference)", lw=1.5)
    plt.plot(x, P_tot, color='#d62728', label=rf"Quantum State $|\psi|^2$ ($\phi={phi_factor:.2f}\pi$)", lw=2.2)
    plt.fill_between(x, P_tot, color='#d62728', alpha=0.2)
    plt.title("Quantum Interference Simulator")
    plt.xlabel("Spatial Coordinate $x$")
    plt.ylabel("Probability Density $P(x)$")
    plt.xlim(-8, 8)
    plt.ylim(0, max(0.6, np.max(P_tot) * 1.15))
    plt.legend(loc="upper right")
    plt.show()

if has_widgets:
    interact(
        interactive_interference,
        d=FloatSlider(min=0.5, max=4.0, step=0.1, value=1.8, description="Separation d:"),
        phi_factor=FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="Phase φ/π:"),
        sigma=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description="Width σ:"),
        k0=FloatSlider(min=0.0, max=3.0, step=0.2, value=1.0, description="Momentum k0:")
    )
else:
    print("Interactive sliders available when running in Jupyter or Google Colab.")
    interactive_interference(d=1.8, phi_factor=1.0, sigma=1.0, k0=1.0)


---
## 6. Hands-On Exercises & Guided Challenges

### Exercise 1: Asymmetric Superposition & Mean Position
Suppose a quantum particle is prepared in an unequal superposition of two localized wavepackets:

$$\psi(x) = \frac{1}{\sqrt{5}} \psi_1(x) + \frac{2}{\sqrt{5}} \psi_2(x)$$

where $\psi_1(x)$ is centered at $x_1 = -2.0$ and $\psi_2(x)$ is centered at $x_2 = +3.0$ (both with width $\sigma = 0.8$ and zero overlap).
1. Predict the analytical expectation value $\langle x \rangle$.
2. Implement this superposition numerically and compute $\langle x \rangle$ using `compute_position_statistics`.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical calculation:**
When the two wavepackets have negligible overlap:
$$\langle x \rangle \approx |c_1|^2 x_1 + |c_2|^2 x_2 = \left(\frac{1}{5}\right)(-2.0) + \left(\frac{4}{5}\right)(3.0) = -0.4 + 2.4 = +2.0$$

**Code snippet:**
```python
psi_ex1_a = gaussian_wavepacket(x, x0=-2.0, sigma=0.8)
psi_ex1_b = gaussian_wavepacket(x, x0=+3.0, sigma=0.8)
psi_ex1 = (1.0/np.sqrt(5.0)) * psi_ex1_a + (2.0/np.sqrt(5.0)) * psi_ex1_b

stats_ex1 = compute_position_statistics(x, psi_ex1)
print(f"Calculated <x>: {stats_ex1['mean_x']:.4f} (Expected: 2.0000)")
```
</details>

---

### Exercise 2: Particle in an Infinite Square Well
The stationary eigenstates for a particle trapped in an infinite potential well $x \in [0, L]$ are:

$$\psi_n(x) = \sqrt{\frac{2}{L}} \sin\left(\frac{n\pi x}{L}\right)$$

Consider an equal superposition of the ground state ($n=1$) and the first excited state ($n=2$):

$$\psi(x) = \frac{1}{\sqrt{2}} \left(\psi_1(x) + \psi_2(x)\right)$$

Construct this state on a grid $x \in [0, L]$ with $L = 5.0$, verify normalization, and plot $P(x)$. Does the probability density remain symmetric about the center $x = L/2$?

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Explanation:**
The ground state $\psi_1(x)$ is symmetric about $L/2$, whereas the first excited state $\psi_2(x)$ is antisymmetric. Their sum breaks spatial inversion symmetry, biasing the probability density toward one side of the box!

**Code snippet:**
```python
L_box = 5.0
x_box = np.linspace(0, L_box, 1000)

psi_n1 = np.sqrt(2.0 / L_box) * np.sin(1 * np.pi * x_box / L_box)
psi_n2 = np.sqrt(2.0 / L_box) * np.sin(2 * np.pi * x_box / L_box)
psi_box = (psi_n1 + psi_n2) / np.sqrt(2.0)

plt.figure(figsize=(8, 4))
plt.plot(x_box, np.abs(psi_box)**2, color="#2ca02c", lw=2, label=r"$|\psi(x)|^2$ Superposition")
plt.axvline(L_box/2, color="gray", ls=":", label="Center L/2")
plt.xlabel("Position $x$")
plt.ylabel("Probability Density")
plt.legend()
plt.show()
```
</details>


---
## 7. Summary & What's Next

### Key Takeaways
1. **Wavefunctions are probability amplitudes:** Physical measurements are governed by Born's rule $P(x) = |\psi(x)|^2$, requiring $L^2$ normalization $\int |\psi(x)|^2 dx = 1$.
2. **Quantum interference distinguishes quantum from classical:** In quantum mechanics, amplitudes sum before squaring ($|\psi_1 + \psi_2|^2 \ne |\psi_1|^2 + |\psi_2|^2$), producing constructive and destructive interference fringes.
3. **Observables are expectation values of operators:** The uncertainty $\Delta x = \sqrt{\langle x^2 \rangle - \langle x \rangle^2}$ measures intrinsic statistical dispersion.
4. **Qubits represent discrete state vectors:** Moving from infinite-dimensional continuous spaces $L^2(\mathbb{R})$ to two-dimensional Hilbert spaces $\mathbb{C}^2$ sets the stage for atomic states, photon polarization, and quantum computing.

---

### What's Next?
In **Notebook 0.2 (`02_qubits_and_density_matrices.ipynb`)**, we will:
* Transition into Dirac bra-ket notation and Pauli matrices using **QuTiP**.
* Contrast pure quantum states with statistical mixtures using **Density Matrices** ($\rho$).
* Compute purity $\mathcal{P} = \text{Tr}(\rho^2)$ and map states directly onto the 3D **Bloch Sphere**.

---
### References & Recommended Reading
* **David J. Griffiths**, *Introduction to Quantum Mechanics*, Chapter 1: The Wave Function.
* **J. J. Sakurai**, *Modern Quantum Mechanics*, Chapter 1: Fundamental Concepts.
* **Christopher Gerry & Peter Knight**, *Introductory Quantum Optics*, Chapter 1.
